In [1]:
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import roc_auc_score

In [2]:
DATASET_ROOT = Path(r"C:\Users\Corey\Downloads\8-facial-expressions-for-yolo\9 Facial Expressions you need")

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 120

MODEL_SAVE_PATH = "expression9_original_style_longrun.keras"

CLASS_NAMES = [
    "angry",
    "contempt",
    "disgust",
    "fear",
    "happy",
    "natural",
    "sad",
    "sleepy",
    "surprised"
]

NUM_CLASSES = len(CLASS_NAMES)

In [3]:
def load_paths_and_labels(split):
    image_folder = DATASET_ROOT / split / "images"
    label_folder = DATASET_ROOT / split / "labels"

    image_paths = []
    labels = []

    for image_path in image_folder.glob("*"):
        if image_path.suffix.lower() not in [".jpg"]:
            continue

        label_path = label_folder / (image_path.stem + ".txt")

        if not label_path.exists():
            continue

        with open(label_path, "r") as file:
            first_line = file.readline().strip()

        class_id = int(first_line.split()[0])

        image_paths.append(str(image_path))
        labels.append(class_id)

    return np.array(image_paths), np.array(labels)

In [4]:
train_paths, train_labels = load_paths_and_labels("train")
valid_paths, valid_labels = load_paths_and_labels("valid")
test_paths, test_labels = load_paths_and_labels("test")

print("Train:", len(train_paths))
print("Valid:", len(valid_paths))
print("Test:", len(test_paths))

Train: 64829
Valid: 1718
Test: 1698


In [5]:
for class_id, class_name in enumerate(CLASS_NAMES):
    count = np.sum(train_labels == class_id)
    print(class_id, class_name, count)

0 angry 11164
1 contempt 2543
2 disgust 4296
3 fear 5358
4 happy 13831
5 natural 5660
6 sad 11966
7 sleepy 1051
8 surprised 8960


In [6]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    return image, label

def make_dataset(paths, labels, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        dataset = dataset.shuffle(len(paths))

    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [7]:
train_dataset = make_dataset(train_paths, train_labels, shuffle=True)
valid_dataset = make_dataset(valid_paths, valid_labels)
test_dataset = make_dataset(test_paths, test_labels)

In [8]:
model = keras.Sequential([
    layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),

    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,194,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │         2,313 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,585,289 (17.49 MB)

 Trainable params: 4,585,289 (17.49 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

In [10]:
checkpoint = keras.callbacks.ModelCheckpoint(
    "expression9_original_style_best.keras",
    monitor="val_loss",
    save_best_only=True
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=6,
    min_lr=0.000001
)

In [11]:
history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=EPOCHS,
    callbacks=[
        checkpoint,
        early_stop,
        reduce_lr
    ]
)

Epoch 1/120
2026/2026 ━━━━━━━━━━━━━━━━━━━━ 226s 111ms/step - accuracy: 0.3597 - loss: 1.6808 - val_accuracy: 0.4627 - val_loss: 1.4197 - learning_rate: 0.0010
Epoch 2/120
2026/2026 ━━━━━━━━━━━━━━━━━━━━ 236s 116ms/step - accuracy: 0.4833 - loss: 1.3720 - val_accuracy: 0.5017 - val_loss: 1.3131 - learning_rate: 0.0010
Epoch 3/120
2026/2026 ━━━━━━━━━━━━━━━━━━━━ 237s 117ms/step - accuracy: 0.5289 - loss: 1.2544 - val_accuracy: 0.5733 - val_loss: 1.1444 - learning_rate: 0.0010
Epoch 4/120
2026/2026 ━━━━━━━━━━━━━━━━━━━━ 238s 117ms/step - accuracy: 0.5590 - loss: 1.1834 - val_accuracy: 0.5978 - val_loss: 1.0838 - learning_rate: 0.0010
Epoch 5/120
2026/2026 ━━━━━━━━━━━━━━━━━━━━ 233s 115ms/step - accuracy: 0.5798 - loss: 1.1261 - val_accuracy: 0.6059 - val_loss: 1.0761 - learning_rate: 0.0010
Epoch 6/120
2026/2026 ━━━━━━━━━━━━━━━━━━━━ 232s 114ms/step - accuracy: 0.6001 - loss: 1.0824 - val_accuracy: 0.6112 - val_loss: 1.0357 - learning_rate: 0.0010
Epoch 7/120
2026/2026 ━━━━━━━━━━━━━━━━━━━━ 233

In [12]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.7256 - loss: 0.7781
Test loss: 0.7781084775924683
Test accuracy: 0.7255594730377197


In [13]:
all_true_labels = []
all_predictions = []

for images, labels in test_dataset:
    predictions = model.predict(images, verbose=0)

    all_true_labels.extend(labels.numpy())
    all_predictions.extend(predictions)

all_true_labels = np.array(all_true_labels)
all_predictions = np.array(all_predictions)

all_true_one_hot = keras.utils.to_categorical(
    all_true_labels,
    num_classes=NUM_CLASSES
)

auc_score = roc_auc_score(
    all_true_one_hot,
    all_predictions,
    average="macro"
)

print("Macro ROC-AUC:", auc_score)

Macro ROC-AUC: 0.9570990745776022


In [14]:
model.save(MODEL_SAVE_PATH)

print("Saved:", MODEL_SAVE_PATH)

Saved: expression9_original_style_longrun.keras
